# gpu

> Explore GPUs available on Modal

In [ ]:
#| default_exp gpu

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| hide 
enable_mermaid()

<script type="module">
if (window.mermaid) mermaid.run()
else {
    import('https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs').then(m => {
        window.mermaid = m.default;
        window.mermaid.run();
        htmx.onLoad(elt => {
            if (elt.matches('div.mermaid, pre.mermaid') || htmx.findAll(elt, 'div.mermaid, pre.mermaid')) window.mermaid.run();
        });
    });
}</script>

In [ ]:
#| export
class GPUEntry:
    def __init__(self,
        tag:str,     # GPU tag, e.g. 'H100'
        price:float, # Price per hour (USD)
        vram:int,    # VRAM in GB
    ) -> None: store_attr()
    def __repr__(self) -> str: return f'{self.tag} — ${self.price:.2f}/hr, {self.vram}GB'
    def _repr_html_(self) -> str: return f'<div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>{self.tag}</b> · ${self.price:.2f}/hr · {self.vram}GB</div>'


In [ ]:
GPUEntry('T4', 0.59, 16)

T4 — $0.59/hr, 16GB

In [ ]:
#| export
from datetime import date

In [ ]:
#| export
class GPU:
    _source = 'https://modal.com/pricing'
    _updated = date(2026, 5, 22)

    T4            = GPUEntry('T4',            0.59,  16)
    L4            = GPUEntry('L4',            0.80,  24)
    A10           = GPUEntry('A10',           1.10,  24)
    L40S          = GPUEntry('L40S',          1.95,  48)
    A100_40GB     = GPUEntry('A100-40GB',     2.10,  40)
    A100_80GB     = GPUEntry('A100-80GB',     2.50,  80)
    RTX_PRO_6000  = GPUEntry('RTX-PRO-6000',  3.03,  96)
    H100          = GPUEntry('H100',          3.95,  80)
    H200          = GPUEntry('H200',          4.54, 141)
    B200          = GPUEntry('B200',          6.25, 180)

    def __repr__(self) -> str:
        lines = [f'GPU variants (as of {self._updated}, from {self._source}):']
        for _, v in sorted(self.__class__.__dict__.items()):
            if isinstance(v, GPUEntry): lines.append(f'  {v.tag:<15} ${v.price:.2f}/hr  {v.vram:>3}GB')
        return '\n'.join(lines)

    def _repr_html_(self) -> str:
        entries = sorted((v for _,v in self.__class__.__dict__.items() if isinstance(v,GPUEntry)), key=lambda e:e.price)
        return '<div style="display:flex;flex-wrap:wrap;gap:8px">' + ''.join(e._repr_html_() for e in entries) + '</div>'


In [ ]:
gpu = GPU(); gpu

GPU variants (as of 2026-05-22, from https://modal.com/pricing):
  A10             $1.10/hr   24GB
  A100-40GB       $2.10/hr   40GB
  A100-80GB       $2.50/hr   80GB
  B200            $6.25/hr  180GB
  H100            $3.95/hr   80GB
  H200            $4.54/hr  141GB
  L4              $0.80/hr   24GB
  L40S            $1.95/hr   48GB
  RTX-PRO-6000    $3.03/hr   96GB
  T4              $0.59/hr   16GB

In [ ]:
#| export
from fastcore.all import *

In [ ]:
#| export
@patch
def cheapest(self:GPU) -> GPUEntry:
    "Return the cheapest GPUEntry."
    return min((v for _,v in self.__class__.__dict__.items() if isinstance(v,GPUEntry)), key=lambda x: x.price)


In [ ]:
gpu.cheapest()

T4 — $0.59/hr, 16GB

In [ ]:
#| export
from IPython.display import HTML

In [ ]:
#| export
@patch
def by_budget(self:GPU,
    max_phr:float,  # Maximum price per hour
) -> HTML:
    "Return GPU cards costing ≤ `max_phr` per hour as HTML."
    entries = sorted((v for _,v in self.__class__.__dict__.items() if isinstance(v,GPUEntry) and v.price <= max_phr), key=lambda e:e.price)
    return HTML('<div style="display:flex;flex-wrap:wrap;gap:8px">' + ''.join(e._repr_html_() for e in entries) + '</div>')


In [ ]:
gpu.by_budget(.8)

HTML(<div style="display:flex;flex-wrap:wrap;gap:8px"><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>T4</b> · $0.59/hr · 16GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>L4</b> · $0.80/hr · 24GB</div></div>)

In [ ]:
#| export
@patch
def price_of(self:GPU,
    tag:str,  # GPU tag, e.g. 'H100'
) -> GPUEntry:
    "Return the GPUEntry matching `tag`."
    for _,v in self.__class__.__dict__.items():
        if isinstance(v,GPUEntry) and v.tag==tag: return v
    tags = [v.tag for _,v in self.__class__.__dict__.items() if isinstance(v,GPUEntry)]
    raise KeyError(f'{tag} not found — available: {tags}')


In [ ]:
gpu.price_of('T4')

T4 — $0.59/hr, 16GB

In [ ]:
#| export
@patch
def tags(self:GPU) -> list[str]:
    "Lists all available GPU tags"
    return [v.tag for _,v in self.__class__.__dict__.items() if isinstance(v,GPUEntry)]

In [ ]:
gpu.tags()

['T4',
 'L4',
 'A10',
 'L40S',
 'A100-40GB',
 'A100-80GB',
 'RTX-PRO-6000',
 'H100',
 'H200',
 'B200']

In [ ]:
#| export
@patch
def most_expensive(self:GPU) -> GPUEntry:
    "Return the most expensive GPUEntry."
    return max((v for _,v in self.__class__.__dict__.items() if isinstance(v,GPUEntry)), key=lambda x: x.price)


In [ ]:
gpu.most_expensive()

B200 — $6.25/hr, 180GB

In [ ]:
#| export
@patch
def by_vram(self:GPU,
    min_gb:int,  # Minimum VRAM in GB
) -> HTML:
    "Return GPU cards with ≥ `min_gb` VRAM as HTML."
    entries = sorted((v for _,v in self.__class__.__dict__.items() if isinstance(v,GPUEntry) and v.vram >= min_gb), key=lambda e:e.price)
    return HTML('<div style="display:flex;flex-wrap:wrap;gap:8px">' + ''.join(e._repr_html_() for e in entries) + '</div>')


In [ ]:
gpu.by_vram(24)

HTML(<div style="display:flex;flex-wrap:wrap;gap:8px"><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>L4</b> · $0.80/hr · 24GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>A10</b> · $1.10/hr · 24GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>L40S</b> · $1.95/hr · 48GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>A100-40GB</b> · $2.10/hr · 40GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>A100-80GB</b> · $2.50/hr · 80GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>RTX-PRO-6000</b> · $3.03/hr · 96GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>H100</b> · $3.95/hr · 80GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>H200</b> · $4.54/hr · 141GB</div><div style="background:#e8f4e8;padding:8px 12px;border-radius:6px;display:inline-block;font-family:monospace"><b>B200</b> · $6.25/hr · 180GB</div></div>)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()